In [ ]:
import numpy as np
import matplotlib.pyplot as plt

student_num = 8151


In [ ]:
fs = 5_000
duration = 2.0
fm = 5.0
fc = 100.0
Am = 1.0
Ac = 1.0
mu = 0.7
snr_db = 10.0

t = np.arange(0, duration, 1 / fs)
print(f'{len(t):,} samples, sample interval = {1/fs:.6f} s')
print(f'Expected AM sidebands: {fc-fm:g} Hz and {fc+fm:g} Hz')


In [ ]:
def one_sided_spectrum(x, fs):
    n = len(x)
    x = np.asarray(x)
    frequencies = np.fft.rfftfreq(n, d=1/fs)
    amplitude = np.abs(np.fft.rfft(x)) * 2 / n
    amplitude[0] /= 2
    if n % 2 == 0:
        amplitude[-1] /= 2
    return frequencies, amplitude


def show_time_and_frequency(x, fs, title, time_window=0.4,
                            max_frequency=200, reference=None,
                            reference_label='Reference'):
    frequencies, amplitude = one_sided_spectrum(x, fs)
    visible = t <= time_window

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    axes[0].plot(t[visible], x[visible], lw=1.3, label=title)

    if reference is not None:
        axes[0].plot(t[visible], reference[visible], '--',
                     lw=1.5, label=reference_label)
        axes[0].legend()

    axes[0].set(title=f'{title} {student_num}: time domain',
                xlabel='Time (s)', ylabel='Amplitude')

    axes[1].plot(frequencies, amplitude, lw=1.1)
    axes[1].set_xlim(0, max_frequency)
    axes[1].set(title=f'{title} {student_num}: frequency domain',
                xlabel='Frequency (Hz)', ylabel='Amplitude spectrum')

    fig.tight_layout()
    plt.show()


In [ ]:
message = Am * np.sin(2 * np.pi * fm * t)

show_time_and_frequency(
    message, fs, 'Message signal',
    time_window=0.8, max_frequency=25
)


In [ ]:
carrier = Ac * np.cos(2 * np.pi * fc * t)

show_time_and_frequency(
    carrier, fs, 'Carrier signal',
    time_window=0.10, max_frequency=160
)


In [ ]:
normalized_message = message / Am
envelope = Ac * (1 + mu * normalized_message)
am_signal = envelope * np.cos(2 * np.pi * fc * t)

frequencies, amplitude = one_sided_spectrum(am_signal, fs)
visible = t <= 0.4

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

axes[0].plot(t[visible], am_signal[visible], lw=0.9, label='AM signal')
axes[0].plot(t[visible], envelope[visible], 'r--', lw=1.5, label='Envelope')
axes[0].plot(t[visible], -envelope[visible], 'r--', lw=1.5)
axes[0].set(
    title=f'AM signal {student_num}: time domain',
    xlabel='Time (s)',
    ylabel='Amplitude'
)
axes[0].legend()

axes[1].plot(frequencies, amplitude, lw=1.1)
axes[1].set_xlim(70, 130)
axes[1].set(
    title=f'AM signal {student_num}: frequency domain',
    xlabel='Frequency (Hz)',
    ylabel='Amplitude spectrum'
)

for frequency, label in [(fc-fm, 'LSB'), (fc, 'Carrier'), (fc+fm, 'USB')]:
    axes[1].axvline(frequency, color='tab:red', ls='--', alpha=0.55)
    axes[1].text(
        frequency, axes[1].get_ylim()[1] * 0.85,
        label, ha='center'
    )

fig.tight_layout()
plt.show()
